In [ ]:
openai_api_key ="YOUR_KEY_HERE"
neo4j_uri = "bolt://localhost:7687"
neo4j_database= "neo4j"
neo4j_username = "neo4j"
neo4j_password = "testtest"

In [1]:
#vector embeddings
#pip install “neo4j_graphrag[openai]

from neo4j import GraphDatabase
from neo4j_graphrag.embeddings import OpenAIEmbeddings
from dotenv import load_dotenv
import os

embeder = OpenAIEmbeddings(model="text-embedding-ada-002", api_key=openai_api_key)

driver = GraphDatabase.driver(neo4j_uri, auth=(neo4j_username, neo4j_password))

#UnstructuredElement
#Chunk

with driver.session() as session:
    res = session.run(
        """
        MATCH (n:Chunk)
        WHERE n.text IS NOT NULL AND n.embedding IS NULL
        RETURN n.id as id, n.text as data
        """
    )
    for record in res:
        chunk_id = record["id"]
        data = record["data"]
        if len(data) > 12000:
            print(f"Skipping chunks {chunk_id} with data length {len(data)}")
            continue
        #print(f"Embedding chunk {chunk_id}")
        print("%s %s" % ('\r embedding chunk:',{chunk_id}), end = "", flush = True)
        embedding = embeder.embed_query(data)
        session.run(
            """
            MATCH (n:Chunk {id: $chunk_id})
            SET n.embedding = $embedding
            """,
            chunk_id=chunk_id,
            embedding=embedding
        )


In [ ]:
#Entity Extraction Prep (run once)

from neo4j import GraphDatabase

# Neo4j connection details
neo4j_uri = "bolt://localhost:7687"
neo4j_user = "neo4j" 
neo4j_password = "testtest"  # Neo4j password

# Neo4j driver setup
driver = GraphDatabase.driver(neo4j_uri, auth=(neo4j_user, neo4j_password))

# to perform selective entity extraction using "ProcessMe" label execute this query

process_me ='''
MATCH (n:Chunk) 
WHERE NOT (n)-[:HAS_ENTITY]->()
AND n.entities IS NULL
SET n:ProcessMe
'''

with driver.session() as session:
    res = session.run(process_me)
session.close()  

In [8]:
#Entity Extraction

from neo4j import GraphDatabase
from openai import OpenAI

client = OpenAI( api_key = openai_api_key)

# Neo4j connection details
neo4j_uri = "bolt://localhost:7687"
neo4j_user = "neo4j" 
neo4j_password = "testtest"  # Neo4j password

# Neo4j driver setup
driver = GraphDatabase.driver(neo4j_uri, auth=(neo4j_user, neo4j_password))


query = '''
MATCH (n:Chunk)-[:PART_OF_DOCUMENT]->(d:Document) 
WHERE NOT (n)-[:HAS_ENTITY]->()
AND n.entities IS NULL
RETURN n.id AS id, replace(n.text,"\n","") AS text LIMIT 1000
'''

query1 = '''
WITH $entities AS entities
MATCH (n:Chunk:ProcessMe {id:$id})
WITH n,entities
CALL apoc.do.when(
    entities[0] = "[]" OR entities[0] STARTS WITH "The text provided does not contain",
    "WITH n SET n.entities = 'failed' REMOVE n:ProcessMe RETURN 0 AS rels",
    "WITH n, apoc.convert.fromJsonList(entities[0]) AS names UNWIND names AS name MERGE (e:Entity {text:name}) WITH n,e MERGE (n)-[:HAS_ENTITY]->(e) WITH DISTINCT n, COUNT(e) AS rels REMOVE n:ProcessMe RETURN rels",
    {n:n,entities:entities}
) YIELD value
RETURN value'''

query2 = '''
MATCH (n:Chunk:ProcessMe {id:$id})
SET n.entities = "failed"
REMOVE n:ProcessMe
'''


def run_exception_query(tx):
    result = tx.run(query2,id=id)
    for record in result:
        print(record)
    return result.consume()


#Customize the prompt as needed, be sure to return results as an array
#Extract all the entities from the following text. Identify only entities, abbreviations and technical terms commonly used in cellular telecommunications networks. Return results as an array. Do not return the entity name if it is not mentioned."

def extract_entities(text):
    prompt = f"""
    Extract all the entities from the following text. Identify only entities, abbreviations and technical terms commonly used in the petroluem exploration, petroleum geology, petroleum reservior analysis and oil & gas production. Return results as an array. Do not return the entity name if it is not mentioned."

    Text: {text}

    """

    messages = [
        {"role": "system", "content": "You help extract information from documents."},
        {"role": "user", "content": prompt}
    ]
    response = client.chat.completions.create(
        model="gpt-4o",  # or GPT-4 if available
        messages=messages,
        max_tokens=500,  # Ensure it fits within token limits
        temperature=0  # Lower temperature for more deterministic results
    )

    data = []
    
    d = response.choices[0].message.content
    
    data.append(d)
    
    return data

count = 0

with driver.session() as session:
    chunks = session.run(query)
    for chunk in chunks:
        
        text = chunk['text']
        id  = chunk['id']
        
        count = count+1
        
        entities = extract_entities(text)
        
        try:
        
            results = session.run(query1,id=id,entities=entities)
            
            for result in results:
                rels = result['value']['rels']
        
                print("%s %s %s %s %s" % ('\r relationships created:',rels,' count:',count,'           '), end = "", flush = True)
            
            results.consume()
        
        except:
            
            summary = session.execute_write(run_exception_query)
        
            continue
            
    chunks.consume()  

    print('done!')

session.close()   

 relationships created: 7  count: 621             done!
